# 05 Modeling — NYC Taxi Zone-Hour Demand



## Modeling Plan

We use a time-aware validation setup because taxi demand is sequential.

- Target: zone-hour taxi demand, usually `n_trips`, `trip_count`, `demand`, `taxi_demand`, or `num_trips`.
- Split: earlier 80% of hours for training, latest 20% for testing.
- Models: Poisson Regression, Random Forest, Gradient Boosting, and optional Ridge baseline.
- Metrics: MAE, RMSE, R², and SMAPE.
- Outputs: saved `.pkl` models, comparison table, plots, and a short modeling summary.

In [ ]:
# Imports and configuration
from pathlib import Path
import warnings
import os

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
# LightGBM is the main performance model for tabular demand data.
from lightgbm import LGBMRegressor

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import PoissonRegressor, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
# Keep matplotlib cache inside the project folder to avoid local permission warnings.
os.environ.setdefault('MPLCONFIGDIR', str(Path.cwd() / '.matplotlib-cache'))

RANDOM_STATE = 42
POLICY_DATE = pd.Timestamp('2025-01-05')
# Keep tuning off for a fast first run; set True for final tuning.
RUN_TUNING = False         # Set True for a slower tuned final run
# Limit CV rows so tuning stays reasonable on a laptop.
MAX_CV_ROWS = 100_000      # Hyperparameter tuning uses the latest training rows
CV_SPLITS = 5
MAX_PLOT_POINTS = 10_000

# Find the project root automatically whether the notebook runs from root or output folder.
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / 'data' / 'processed').exists()
)
PROC_DIR = PROJECT_ROOT / 'data' / 'processed'
MODEL_DIR = PROJECT_ROOT / 'models'
OUT_DIR = PROJECT_ROOT / 'outputs' / 'modeling'
# Create output folders before saving models and plots.
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Processed data:', PROC_DIR)
print('Models output:', MODEL_DIR)
print('Modeling output:', OUT_DIR)
print('Boosting backend: LightGBM')

## 1. Load Or Create Zone-Hour Feature Matrix

The modeling input is `data/processed/feature_matrix_demand.parquet`, created by the feature-engineering section.

In [ ]:
# Modeling starts from the zone-hour demand matrix created in feature engineering.
feature_path = PROC_DIR / 'feature_matrix_demand.parquet'
if not feature_path.exists():
    raise FileNotFoundError(
        f'Missing {feature_path}. Run the feature engineering section first to create it.'
    )

print('Using feature matrix:', feature_path)
df = pd.read_parquet(feature_path)
print('Shape:', df.shape)
df.head()

## 2. Identify Target And Time Column

The target should be the count of taxi trips for a zone-hour observation. The time column is used only for time-based splitting and is not used as a raw model feature.

In [ ]:
# The target should be the zone-hour taxi trip count.
target_candidates = ['n_trips', 'trip_count', 'demand', 'taxi_demand', 'num_trips']
# The time column is needed for chronological train/test splitting.
time_candidates = ['hour_ts', 'pickup_hour_ts', 'datetime', 'timestamp', 'pickup_datetime', 'hour']

target_col = next((c for c in target_candidates if c in df.columns), None)
time_col = next((c for c in time_candidates if c in df.columns), None)

# Fail early if the feature matrix does not contain a clear demand target.
if target_col is None:
    raise ValueError(f'No demand target found. Tried {target_candidates}. Columns are: {df.columns.tolist()}')
if time_col is None:
    datetime_cols = list(df.select_dtypes(include=['datetime64[ns]', 'datetime64[us]']).columns)
    if datetime_cols:
        time_col = datetime_cols[0]
    else:
        raise ValueError(f'No time column found. Tried {time_candidates}. Columns are: {df.columns.tolist()}')

# Convert the time column before sorting and splitting.
if not pd.api.types.is_datetime64_any_dtype(df[time_col]):
    df[time_col] = pd.to_datetime(df[time_col], errors='coerce')

print('Target column:', target_col)
print('Time column:', time_col)
print('Target summary:')
df[target_col].describe()

## 3. Time-Based Train/Test Split

We sort observations by hour and train on earlier hours. The test set is the latest 20% of the project window, which avoids random time leakage.

<!-- modeling-notation:time-split -->
### Time Split Note

The split is chronological rather than random. The model trains on earlier hours and tests on later hours, so test performance better reflects the real task: predicting future taxi demand after observing past demand patterns.

In [ ]:
# Remove rows missing target or time before modeling.
df_model = df.dropna(subset=[time_col, target_col]).sort_values(time_col).reset_index(drop=True)
unique_times = np.array(sorted(df_model[time_col].dropna().unique()))
# Train on the first 80% of timestamps and test on the latest 20%.
cutoff_time = unique_times[int(len(unique_times) * 0.80)]

train_df = df_model[df_model[time_col] < cutoff_time].copy()
test_df = df_model[df_model[time_col] >= cutoff_time].copy()

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print('Train time range:', train_df[time_col].min(), 'to', train_df[time_col].max())
print('Test time range:', test_df[time_col].min(), 'to', test_df[time_col].max())

## 4. Prepare Features

We remove direct leakage columns. Lag and rolling demand features are kept because they are past information. Current-hour aggregate trip statistics such as `n_card` and `avg_fare_missing` are removed because they are only known after the hour is complete.

In [ ]:
# Past-demand markers identify safe lag and rolling features.
past_markers = ['lag', 'ma_', 'rolling', 'past', 'prev']
# Drop columns that would reveal current-hour demand after the fact.
leakage_words = [
    'future', 'target', 'actual', 'total_trips', 'total_demand',
    'n_card', 'card_count', 'avg_fare', 'avg_distance', 'avg_duration', 'avg_speed'
]
target_like_words = ['n_trips', 'trip_count', 'taxi_demand', 'num_trips', 'demand']


# Keep lag and rolling demand variables because they use past information only.
def is_past_feature(col):
    c = col.lower()
    return any(marker in c for marker in past_markers)


def choose_feature_columns(data, target_col, time_col):
    drop_cols = {target_col, time_col}
    for col in data.columns:
        c = col.lower()
        if col in drop_cols:
            continue
        if pd.api.types.is_datetime64_any_dtype(data[col]) or c in ['date', 'pickup_date']:
            drop_cols.add(col)
        elif any(word in c for word in leakage_words):
            drop_cols.add(col)
        elif any(word in c for word in target_like_words) and not is_past_feature(c):
            drop_cols.add(col)
    return [c for c in data.columns if c not in drop_cols], sorted(drop_cols)


# Finalize usable model columns after leakage filtering.
feature_cols, dropped_cols = choose_feature_columns(df_model, target_col, time_col)

X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()
y_train = train_df[target_col].astype(float)
y_test = test_df[target_col].astype(float)

# Zone IDs and cluster labels are categorical, not continuous measurements.
for col in ['PULocationID', 'pickup_zone_id', 'zone_id', 'zone_cluster', 'borough', 'PUBorough']:
    if col in X_train.columns:
        X_train[col] = X_train[col].astype('category')
        X_test[col] = X_test[col].astype('category')

# Convert booleans to numeric flags for sklearn pipelines.
bool_cols = X_train.select_dtypes(include=['bool']).columns
X_train[bool_cols] = X_train[bool_cols].astype('int8')
X_test[bool_cols] = X_test[bool_cols].astype('int8')

# Separate categorical and numeric columns for different preprocessing steps.
categorical_cols = list(X_train.select_dtypes(include=['object', 'category']).columns)
numeric_cols = [c for c in X_train.columns if c not in categorical_cols]

for col in numeric_cols:
    X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce')

print('Number of features:', len(feature_cols))
print('Numeric features:', len(numeric_cols))
print('Categorical features:', len(categorical_cols))
print('Dropped columns:', dropped_cols)
X_train.head()

## 5. Build Modeling Pipelines

Poisson and Ridge use scaled numeric features. Tree models use imputed numeric features and one-hot encoded categorical variables.

In [ ]:
def make_preprocessor(scale_numeric=False):
    # Median imputation handles missing numeric values inside the pipeline.
    numeric_steps = [('imputer', SimpleImputer(strategy='median'))]
    # Poisson and Ridge benefit from scaled numeric features.
    if scale_numeric:
        numeric_steps.append(('scaler', StandardScaler()))

    # One-hot encoding turns categories into model-ready columns.
    categorical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=np.float32))
    ])

    return ColumnTransformer([
        ('num', Pipeline(numeric_steps), numeric_cols),
        ('cat', categorical_pipe, categorical_cols)
    ], remainder='drop')


# LightGBM is the gradient boosting model used for final comparison.
boosting_model = LGBMRegressor(
    objective='regression',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

models = {
    # Poisson is the interpretable count-data baseline.
    'Poisson': Pipeline([
        ('preprocess', make_preprocessor(scale_numeric=True)),
        ('model', PoissonRegressor(alpha=0.001, max_iter=1000))
    ]),
    'Ridge': Pipeline([
        ('preprocess', make_preprocessor(scale_numeric=True)),
        ('model', Ridge(alpha=1.0, random_state=RANDOM_STATE))
    ]),
    # Random Forest is the nonlinear tree-based baseline.
    'RandomForest': Pipeline([
        ('preprocess', make_preprocessor(scale_numeric=False)),
        ('model', RandomForestRegressor(
            n_estimators=150,
            max_depth=20,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ]),
    # LightGBM is expected to perform well on structured tabular features.
    'LightGBM': Pipeline([
        ('preprocess', make_preprocessor(scale_numeric=False)),
        ('model', boosting_model)
    ])
}

models.keys()

## 6. Light Hyperparameter Tuning And Training

Only Random Forest and Boosting are tuned. The search is intentionally small so the notebook remains practical for a school final project.

In [ ]:
# Small parameter grids keep tuning appropriate for a school project.
rf_params = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [10, 20, None],
    'model__min_samples_leaf': [1, 5],
}

# LightGBM tuning focuses on learning rate, tree size, and number of trees.
lightgbm_params = {
    'model__learning_rate': [0.03, 0.06, 0.1],
    'model__n_estimators': [100, 200],
    'model__num_leaves': [15, 31],
    'model__max_depth': [-1, 10, 20],
}

param_grids = {
    'RandomForest': rf_params,
    'LightGBM': lightgbm_params,
}

# Tune on recent training rows to reduce runtime while preserving time order.
X_cv = X_train.tail(min(MAX_CV_ROWS, len(X_train)))
y_cv = y_train.tail(min(MAX_CV_ROWS, len(y_train)))
# TimeSeriesSplit avoids random leakage in sequential demand data.
ts_cv = TimeSeriesSplit(n_splits=CV_SPLITS)

fitted_models = {}
for name, model in models.items():
    print(f'\nTraining {name}...')
    # Only RandomForest and LightGBM are tuned; simple baselines use fixed settings.
    if RUN_TUNING and name in param_grids:
        search = RandomizedSearchCV(
            estimator=model,
            param_distributions=param_grids[name],
            n_iter=6,
            scoring='neg_mean_absolute_error',
            cv=ts_cv,
            n_jobs=-1,
            random_state=RANDOM_STATE,
            verbose=1,
        )
        search.fit(X_cv, y_cv)
        print('Best params:', search.best_params_)
        best_model = search.best_estimator_
        best_model.fit(X_train, y_train)
        fitted_models[name] = best_model
    else:
        model.fit(X_train, y_train)
        fitted_models[name] = model

print('\nFinished training models:', list(fitted_models.keys()))

## 7. Evaluate Models

MAE and RMSE are the main performance metrics because they are easy to interpret as trips per zone-hour. R² is included for model fit, and SMAPE gives a percentage-style error that is safer around zero demand than MAPE.

In [ ]:
# SMAPE is safer than MAPE when actual demand can be zero.
def smape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.abs(y_true) + np.abs(y_pred)
    denom = np.where(denom == 0, 1, denom)
    return np.mean(2 * np.abs(y_pred - y_true) / denom) * 100


def evaluate(name, model):
    # Demand cannot be negative, so predictions are clipped at zero.
    pred = np.clip(model.predict(X_test), 0, None)
    return {
        'model': name,
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'R2': r2_score(y_test, pred),
        'SMAPE': smape(y_test, pred),
    }, pred

results = []
predictions = {}
for name, model in fitted_models.items():
    row, pred = evaluate(name, model)
    results.append(row)
    predictions[name] = pred

# Sort model results by MAE and RMSE for selection.
comparison = pd.DataFrame(results).sort_values(['MAE', 'RMSE']).reset_index(drop=True)
comparison.to_csv(OUT_DIR / 'model_comparison.csv', index=False)
comparison

## 8. Select Final Model

The final model is selected primarily by MAE and RMSE on the latest 20% time-based test set. This favors the model that predicts future demand most accurately while still keeping interpretation manageable.

In [ ]:
# Choose the best model from the time-based test set.
final_model_name = comparison.iloc[0]['model']
final_model = fitted_models[final_model_name]
final_pred = predictions[final_model_name]
final_metrics = comparison[comparison['model'] == final_model_name].iloc[0]

print('Selected final model:', final_model_name)
print(final_metrics)

# Save each trained model so teammates can reuse them.
joblib.dump(fitted_models['Poisson'], MODEL_DIR / 'poisson_model.pkl')
joblib.dump(fitted_models['RandomForest'], MODEL_DIR / 'random_forest_model.pkl')
joblib.dump(fitted_models['LightGBM'], MODEL_DIR / 'boosting_model.pkl')
joblib.dump(final_model, MODEL_DIR / 'final_model.pkl')
if 'Ridge' in fitted_models:
    joblib.dump(fitted_models['Ridge'], MODEL_DIR / 'ridge_model.pkl')

print('Saved models to:', MODEL_DIR)

## 9. Evaluation Plots

These plots can be used directly in the final report or presentation.

In [ ]:
# Use a fixed random sample so plots are reproducible.
rng = np.random.default_rng(RANDOM_STATE)
n_plot = min(MAX_PLOT_POINTS, len(y_test))
# Plot a sample to keep scatter plots readable.
plot_idx = rng.choice(len(y_test), size=n_plot, replace=False)
y_plot = y_test.to_numpy()[plot_idx]
pred_plot = final_pred[plot_idx]

plt.figure(figsize=(7, 6))
plt.scatter(y_plot, pred_plot, alpha=0.25, s=10)
limit = max(y_plot.max(), pred_plot.max())
plt.plot([0, limit], [0, limit], color='red', linestyle='--')
plt.xlabel('Actual demand')
plt.ylabel('Predicted demand')
plt.title(f'Actual vs Predicted Demand — {final_model_name}')
plt.tight_layout()
plt.savefig(OUT_DIR / 'actual_vs_predicted.png', dpi=150)
plt.show()

plt.figure(figsize=(7, 5))
# Residuals show where the final model over- or under-predicts demand.
residuals = y_plot - pred_plot
plt.scatter(pred_plot, residuals, alpha=0.25, s=10)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicted demand')
plt.ylabel('Residual')
plt.title(f'Residual Plot — {final_model_name}')
plt.tight_layout()
plt.savefig(OUT_DIR / 'residual_plot.png', dpi=150)
plt.show()

## 10. Feature Importance

For Random Forest, we use built-in feature importance. For Boosting, Poisson, or Ridge, we use permutation importance on a sample of the test set.

In [ ]:
def get_feature_importance(model, model_name):
    regressor = model.named_steps['model']

    # Tree models have built-in importance; otherwise use permutation importance.
    if hasattr(regressor, 'feature_importances_'):
        feature_names = model.named_steps['preprocess'].get_feature_names_out()
        importance = regressor.feature_importances_
        importance_df = pd.DataFrame({'feature': feature_names, 'importance': importance})
    else:
        # Limit permutation importance sample size to keep runtime manageable.
        sample_n = min(5_000, len(X_test))
        X_sample = X_test.sample(sample_n, random_state=RANDOM_STATE)
        y_sample = y_test.loc[X_sample.index]
        result = permutation_importance(
            model,
            X_sample,
            y_sample,
            n_repeats=5,
            scoring='neg_mean_absolute_error',
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        importance_df = pd.DataFrame({
            'feature': X_test.columns,
            'importance': result.importances_mean,
        })

    return importance_df.sort_values('importance', ascending=False).head(20)


importance_df = get_feature_importance(final_model, final_model_name)
importance_df.to_csv(OUT_DIR / 'feature_importance.csv', index=False)

plt.figure(figsize=(9, 6))
plt.barh(importance_df['feature'][::-1], importance_df['importance'][::-1])
plt.xlabel('Importance')
plt.title(f'Top Feature Importance — {final_model_name}')
plt.tight_layout()
plt.savefig(OUT_DIR / 'feature_importance.png', dpi=150)
plt.show()

importance_df

## 11. Save Modeling Summary

This text file is written so the modeling results can be copied into the final report.

In [ ]:
# Save top predictors for the written modeling summary.
top_features = ', '.join(importance_df['feature'].astype(str).head(8))

metrics_text = (
    f"Final model: {final_model_name}\n"
    f"MAE: {final_metrics['MAE']:.4f}\n"
    f"RMSE: {final_metrics['RMSE']:.4f}\n"
    f"R2: {final_metrics['R2']:.4f}\n"
    f"SMAPE: {final_metrics['SMAPE']:.2f}%\n"
)
(OUT_DIR / 'final_model_metrics.txt').write_text(metrics_text)

summary_text = (
    "NYC Taxi Demand Modeling Summary\n\n"
    "Goal:\n"
    "Predict hourly taxi demand by pickup zone and evaluate how time, location, weather, "
    "and congestion-pricing policy features help explain demand.\n\n"
    f"Input feature matrix:\n{feature_path}\n\n"
    "Dataset:\n"
    f"Rows: {len(df_model):,}\n"
    f"Columns: {df_model.shape[1]:,}\n"
    f"Target: {target_col}\n"
    f"Time column: {time_col}\n"
    f"Training window: {train_df[time_col].min()} to {train_df[time_col].max()}\n"
    f"Testing window: {test_df[time_col].min()} to {test_df[time_col].max()}\n\n"
    "Models compared:\n"
    f"{comparison.to_string(index=False)}\n\n"
    "Selected final model:\n"
    f"{final_model_name}\n\n"
    "Reason for selection:\n"
    "The final model was selected using the time-based test set, prioritizing MAE and RMSE. "
    "This is more realistic than a random split because the project is about predicting future "
    "taxi demand during a policy change period.\n\n"
    "Most important predictors:\n"
    f"{top_features}\n\n"
    "Interpretation for congestion-pricing story:\n"
    "Important time features show predictable commuting and nightlife demand patterns. "
    "Spatial features such as borough, pickup zone, CBD/treatment indicators, and zone clusters "
    "capture where demand is concentrated. Policy features such as post_policy, treated_zone, "
    "and did help compare post-policy demand changes for CBD-related zones versus other areas. "
    "Weather and holiday variables control for short-term demand shocks.\n\n"
    "Limitations:\n"
    "- The model is predictive, not a complete causal estimate of congestion pricing.\n"
    "- Results depend on the quality of zone-hour aggregation and lag features.\n"
    "- Lag features are strong predictors but may dominate policy variables.\n"
    "- Citi Bike features are optional and only included if they exist in the feature matrix.\n"
)
(OUT_DIR / 'modeling_summary.txt').write_text(summary_text)

print(metrics_text)
print('Saved outputs to:', OUT_DIR)

## 12. Simple Prediction Interface

This function gives teammates a small `predict(X)` style interface for stacking, presentation demos, or a Streamlit app.

In [ ]:
# Simple wrapper for using the saved final model in a demo or app.
def predict_demand(new_data):
    """Predict zone-hour taxi demand using the saved final model."""
    model = joblib.load(MODEL_DIR / 'final_model.pkl')
    return np.clip(model.predict(new_data), 0, None)

print('Example usage: predictions = predict_demand(X_test.head())')
print(predict_demand(X_test.head()))

## Modeling Deliverables

After running the notebook, the main files are:

- `models/poisson_model.pkl`
- `models/random_forest_model.pkl`
- `models/boosting_model.pkl`
- `models/final_model.pkl`
- `outputs/modeling/model_comparison.csv`
- `outputs/modeling/final_model_metrics.txt`
- `outputs/modeling/actual_vs_predicted.png`
- `outputs/modeling/residual_plot.png`
- `outputs/modeling/feature_importance.png`
- `outputs/modeling/modeling_summary.txt`